<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2001%20-%20What%20Does%20It%20Mean%20for%20a%20Machine%20to%20Learn/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 01 — What Does It Mean for a Machine to Learn? · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook does not repeat it; it *tests* it.

## Step 1 — The Problem

Four houses have sold. Rooms and price (₹ lakh):

| 1 | 2 | 3 | 4 |
|---|---|---|---|
| 3 | 5 | 7 | 9 |

**A five-room house comes on the market. What should we ask for it?**

Nobody gives the machine the pricing rule. It has four facts and must find the rule itself.

## Step 2 — Prediction

Commit to answers **before running anything**. Write them down; you will check them in Step 10.

1. Starting from $w=0$, $b=0$, what values will $w$ and $b$ end at?
2. What price will the machine quote for five rooms?
3. With a learning rate of $0.001$ instead of $0.01$, does training fail — and if so, *how*?
4. With a learning rate of $0.13$, what exactly goes wrong? A wrong answer, or something else?

Being wrong here is useful. Being vague is not.

In [ ]:
# Step 3 — Intuition: turn the dials by hand.
import numpy as np
np.random.seed(0)

rooms  = np.array([1., 2., 3., 4.])   # x — the input
prices = np.array([3., 5., 7., 9.])   # y — what the houses actually sold for

def predict(w, b, x=rooms):
    """The model from blog section 4:  y_hat = w*x + b"""
    return w * x + b

# Three settings of the dials. Which looks closest to the real prices?
for w, b in [(1.0, 0.0), (0.0, 6.0), (2.0, 1.0)]:
    print(f"w={w}, b={b}  ->  predictions {predict(w, b)}   actual {prices}")

## Step 4 — The Mathematics Under Test

Three equations from the lecture, and nothing else:

$$\hat{y} = wx + b
\qquad
L = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2
\qquad
\theta \leftarrow \theta - \eta \nabla_\theta L$$

Step 5 checks every number the lecture claims. If one of these `assert`s ever fails, the
lecture and the laboratory have drifted apart and one of them is wrong.

In [ ]:
# Step 5 — Manual calculation: verify every numeric claim in blog.md.

def mse(w, b, x=rooms, y=prices):
    """Mean squared error — blog section 8."""
    return np.mean((predict(w, b, x) - y) ** 2)

# --- section 7: why averaging signed errors fails --------------------------
errors_flat = predict(0.0, 6.0) - prices          # the "₹6 lakh for everyone" model
print("errors:", errors_flat)
print(f"mean signed error = {errors_flat.mean():.1f}   <- looks perfect, model is useless")
print(f"mean absolute     = {np.abs(errors_flat).mean():.1f}")
print(f"mean squared      = {(errors_flat ** 2).mean():.1f}")
assert errors_flat.mean() == 0.0          # signed errors cancel exactly
assert np.abs(errors_flat).mean() == 2.0
assert mse(0.0, 6.0) == 5.0

# --- section 8: the loss ranks the models the way we do -------------------
assert mse(2.0, 0.0) == 1.0               # right slope, forgot the land
assert mse(2.0, 1.0) == 0.0               # the truth

# --- section 10: the loss landscape, with b pinned to 1 -------------------
print("\n  w      L(w)    7.5(w-2)^2")
for w in [0.0, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0]:
    closed_form = 7.5 * (w - 2) ** 2
    print(f"{w:5.1f} {mse(w, 1.0):8.3f} {closed_form:11.3f}")
    assert abs(mse(w, 1.0) - closed_form) < 1e-12

assert np.mean(rooms ** 2) == 7.5         # the 30/4 that produced the 7.5

In [ ]:
# Step 5b — the slope we derived by hand in section 11, checked numerically.
# We claimed:  slope of L at w  =  15(w - 2)

def loss_1d(w):
    return 7.5 * (w - 2) ** 2

for w in [0.0, 1.0, 3.0]:
    analytic = 15 * (w - 2)
    for h in [1e-2, 1e-4, 1e-6]:
        numerical = (loss_1d(w + h) - loss_1d(w)) / h
        print(f"w={w}  h={h:<8} numerical={numerical:9.5f}   analytic={analytic}")
    # h = 1e-6: close enough that the leftover 7.5h term is tiny, large enough
    # that subtracting two nearly equal numbers stays numerically safe.
    assert abs((loss_1d(w + 1e-6) - loss_1d(w)) / 1e-6 - analytic) < 1e-4
    print()

# The sign is the compass: negative means "increase w", positive means "decrease w".

In [ ]:
# Step 5c — one full step of learning, matching the hand calculation in section 13.
w, b, learning_rate = 0.0, 0.0, 0.01

y_hat = predict(w, b)
error = y_hat - prices
loss_before = np.mean(error ** 2)

dw = np.mean(2 * error * rooms)     # dL/dw
db = np.mean(2 * error)             # dL/db

w_new = w - learning_rate * dw
b_new = b - learning_rate * db
loss_after = mse(w_new, b_new)

print(f"loss before = {loss_before}")
print(f"dw = {dw},  db = {db}      (both negative: turn both dials up)")
print(f"w: {w} -> {w_new},   b: {b} -> {b_new}")
print(f"loss after  = {loss_after:.5f}")

assert loss_before == 41.0
assert dw == -35.0 and db == -12.0
assert abs(w_new - 0.35) < 1e-12 and abs(b_new - 0.12) < 1e-12
assert abs(loss_after - 28.45315) < 1e-5

In [ ]:
# Step 6 — First implementation: the whole training loop, from scratch.
# No framework, no optimizer object, no .fit(). Just the four lines of section 13.

def train(x, y, learning_rate=0.01, steps=2000, w=0.0, b=0.0, record=False):
    history = []
    for step in range(steps):
        y_hat = w * x + b                       # 1. predict
        error = y_hat - y                       # 2. measure
        loss  = np.mean(error ** 2)
        dw    = np.mean(2 * error * x)          # 3. slopes
        db    = np.mean(2 * error)
        w -= learning_rate * dw                 # 4. step downhill
        b -= learning_rate * db
        if record:
            history.append((w, b, loss))
        if not np.isfinite(w) or abs(w) > 1e12:  # it exploded; stop early
            break
    return w, b, history

w, b, history = train(rooms, prices, learning_rate=0.01, steps=2000, record=True)
five_rooms = w * 5 + b

print(f"learned w = {w:.4f}   (₹ lakh per room — nobody told it this)")
print(f"learned b = {b:.4f}   (₹ lakh for the plot)")
print(f"final loss = {mse(w, b):.8f}")
print(f"\nPrice for a five-room house: ₹{five_rooms:.2f} lakh")

assert abs(w - 2.0) < 1e-2 and abs(b - 1.0) < 1e-2
assert abs(five_rooms - 11.0) < 1e-2

In [ ]:
# Step 6b — code mirrors the mathematics.
# The vectorized line hides a sum. Here is the same gradient written out the way
# the formula in section 11 reads, one house at a time:
#
#     dL/dw = (1/n) * sum over i of  2 * (y_hat_i - y_i) * x_i

def gradient_explicit(w, b, x, y):
    n = len(x)
    total = 0.0
    for i in range(n):                      # what the sigma actually says
        y_hat_i = w * x[i] + b
        total += 2 * (y_hat_i - y[i]) * x[i]
    return total / n

def gradient_vectorized(w, b, x, y):
    return np.mean(2 * (w * x + b - y) * x)  # what you write once you understand it

for test_w, test_b in [(0.0, 0.0), (1.5, 0.5), (2.0, 1.0)]:
    a = gradient_explicit(test_w, test_b, rooms, prices)
    c = gradient_vectorized(test_w, test_b, rooms, prices)
    print(f"w={test_w}, b={test_b}:  explicit={a:8.4f}   vectorized={c:8.4f}")
    assert abs(a - c) < 1e-12

# Identical. The vectorized form exists for speed, not for magic —
# optimized libraries run that loop in compiled code, in parallel.

In [ ]:
# Step 7 — Visualization: the three pictures the lecture predicted.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) the fitted line against the four houses
axes[0].scatter(rooms, prices, s=70, zorder=3, label="actual sales")
grid = np.linspace(0, 5.5, 100)
axes[0].plot(grid, w * grid + b, color="crimson", label=f"learned: {w:.2f}x + {b:.2f}")
axes[0].scatter([5], [five_rooms], marker="*", s=260, color="darkorange",
                zorder=3, label=f"5 rooms -> ₹{five_rooms:.1f} lakh")
axes[0].set_xlabel("rooms"); axes[0].set_ylabel("price (₹ lakh)")
axes[0].set_title("The rule it discovered")

# (b) loss falling as it learns
losses = [h[2] for h in history]
axes[1].plot(losses)
axes[1].set_yscale("log")
axes[1].set_xlabel("step"); axes[1].set_ylabel("loss (log scale)")
axes[1].set_title("Loss during training")

# (c) the landscape of section 10, with the descent path on it (b pinned to 1)
w_grid = np.linspace(-0.5, 4.5, 200)
axes[2].plot(w_grid, 7.5 * (w_grid - 2) ** 2, label="L(w) = 7.5(w-2)²")
path_w, path_L = [], []
wi = 0.0
for _ in range(20):
    path_w.append(wi); path_L.append(7.5 * (wi - 2) ** 2)
    wi = wi - 0.01 * 15 * (wi - 2)            # step against the slope
axes[2].plot(path_w, path_L, "o-", color="crimson", ms=4, label="downhill steps")
axes[2].set_xlabel("w"); axes[2].set_ylabel("loss")
axes[2].set_title("Walking down the valley")

for ax in axes:
    ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# The descent on a parabola shrinks the distance to the bottom by a constant
# factor every step: (w - 2) <- 0.85(w - 2) when eta = 0.01. Check it:
wi = 0.0
for _ in range(5):
    w_next = wi - 0.01 * 15 * (wi - 2)
    assert abs((w_next - 2) - 0.85 * (wi - 2)) < 1e-12
    wi = w_next
print("confirmed: each step closes 15% of the remaining distance to w = 2")

In [ ]:
# Step 8 — The controlled experiment.
# Everything is held fixed: same data, same start (w=0, b=0), same 200 steps.
# Only the learning rate changes.

def run(learning_rate, steps=200):
    w, b, _ = train(rooms, prices, learning_rate=learning_rate, steps=steps)
    if not np.isfinite(w) or abs(w) > 1e12:
        return w, b, float("inf")
    return w, b, mse(w, b)

w_ref, b_ref, loss_ref = run(0.01)
print(f"reference run (eta=0.01): w={w_ref:.4f}, b={b_ref:.4f}, loss={loss_ref:.6f}")

In [ ]:
# Step 9 — Change exactly one variable: eta. This reproduces the table in section 14.
print(f"{'eta':>7} {'w':>12} {'b':>12} {'loss':>14}   verdict")
for eta in [0.001, 0.01, 0.05, 0.11, 0.12, 0.13]:
    with np.errstate(over="ignore", invalid="ignore"):
        w_e, b_e, loss_e = run(eta)
    if not np.isfinite(loss_e):
        print(f"{eta:>7} {'exploded':>12} {'':>12} {'inf':>14}   diverged")
    else:
        # 41.0 is the loss we started from, so anything above it is going backwards.
        verdict = "working" if loss_e < 0.1 else ("diverging" if loss_e > 41.0 else "too slow")
        print(f"{eta:>7} {w_e:12.4f} {b_e:12.4f} {loss_e:14.6f}   {verdict}")

# Watch the failure happen, step by step, at eta = 0.13:
print("\nfirst five steps at eta = 0.13:")
w_d, b_d = 0.0, 0.0
for step in range(5):
    err = (w_d * rooms + b_d) - prices
    print(f"  step {step}: w = {w_d:8.2f}   loss = {np.mean(err ** 2):10.2f}")
    w_d -= 0.13 * np.mean(2 * err * rooms)
    b_d -= 0.13 * np.mean(2 * err)

## Step 10 — Observe

Compare against the four predictions you wrote in Step 2.

What the run shows:

- $w \to 2$ and $b \to 1$ — the machine recovered "₹2 lakh per room, ₹1 lakh for the plot"
  from four sales, having been told neither number.
- Five rooms → **₹11 lakh**.
- At $\eta = 0.001$ nothing breaks; it is simply still crawling after 200 steps, with $b$
  stuck near $0.7$. **Slowness is a failure mode too.**
- At $\eta = 0.13$ the loss does not settle on a wrong answer — it *grows every step*, and
  $w$ flips sign each time: $0 \to 4.55 \to -0.79 \to 5.46$. It is bouncing between the walls
  of the valley, climbing higher with every bounce, until the numbers overflow.

## Step 11 — Explain

Why does it explode at $0.13$ but not at $0.11$? Not bad luck — arithmetic.

From section 10, the steepness of our valley is set by $\frac{1}{n}\sum x_i^2 = 7.5$. A
gradient step multiplies the distance-to-the-bottom by a fixed factor each time. For the
one-dial case in Step 7 that factor was $1 - 0.15 = 0.85$: each step closes 15% of the gap,
so the distance shrinks geometrically and we converge.

Raise $\eta$ and that factor grows. Once it passes $1$, every step *increases* the distance
instead of shrinking it, and the increase compounds — which is exactly the sign-flipping
blow-up printed above. For this dataset the turning point sits near $\eta \approx 0.12$, and
the experiment breaks between $0.11$ and $0.12$.

> The mathematics predicted where the machine would break, before the machine broke.
> That is the whole reason to learn the mathematics.

Note what this also means: the threshold depends on the *data*, through $\sum x_i^2$. Change
the units of your input and the safe learning rate changes with it — which is Challenge 1.

In [ ]:
# Step 12 — Challenges.

# LEVEL 4 (Investigate), part 1:
# Find the largest eta that still converges, to two decimal places.
# Hint: scan upward and watch where loss stops being finite.

# YOUR CODE HERE


# LEVEL 4 (Investigate), part 2:
# Now measure rooms in *hundreds of square feet* instead: x = [10, 20, 30, 40],
# same prices. Find the stable eta threshold again — then explain the shift
# using mean(x**2) from section 10.
rooms_rescaled = np.array([10., 20., 30., 40.])
print("mean(x^2) was", np.mean(rooms ** 2), "-> now", np.mean(rooms_rescaled ** 2))

# YOUR CODE HERE


# LEVEL 5 (Design):
# Squared error treats overcharging and undercharging as identical mistakes.
# Write a loss that punishes overcharging (y_hat > y) more heavily, implement it,
# train with it, and describe how the learned line moves. Keep it smooth — section 11
# explains why a kink would be a problem.

def asymmetric_loss(y_hat, y, penalty=2.0):
    # YOUR CODE HERE
    ...

## Step 13 — Reflection

- [ ] I predicted the outcomes in Step 2 *before* running anything.
- [ ] I can point to the line of code that is $\theta \leftarrow \theta - \eta\nabla_\theta L$.
- [ ] I can explain why `errors_flat.mean() == 0` for a model that is obviously bad.
- [ ] I derived $15(w-2)$ by hand and watched Step 5b confirm it numerically.
- [ ] I can say what breaks at $\eta = 0.13$ — and why it is oscillation, not drift.
- [ ] Nothing here was called `learn()`. I can name the four arithmetic steps that did it.

### The question this chapter leaves open

We described an entire house with **one number**. Real houses have area, age, floor and
distance to the station; a photograph has millions of pixels.

The moment a house needs three measurements, $\hat{y} = wx + b$ runs out — we need a way to
hold many numbers as one object and multiply many dials against them at once.

➡️ **Next:** [Chapter 02 — Numbers Become Vectors](<../Lecture 02 - Numbers Become Vectors/blog.md>)